# 01 — Quality control

Decides, in code, which participants and trials enter the analysis set, so that
the Methods section can cite a rule instead of a judgement call.

Reads `data/derived/` (the MATLAB output tables) and `data/raw/vr/` (read-only).
Writes `data/derived/qc_trial_inventory.csv` and `data/derived/qc_exclusions.csv`.

*Определяет кодом, кто и какие трайлы попадают в анализ, чтобы в Methods можно
было сослаться на правило, а не на решение «на глаз». Читает `data/derived/` и
`data/raw/vr/` (только на чтение), пишет два QC-файла в `data/derived/`.*

### Setup

Paths, acquisition constants and the QC thresholds. The thresholds live here and
nowhere else, so the text below can refer to them by name and cannot drift from
what the code actually uses.

*Пути, константы записи и пороги QC. Пороги заданы здесь и больше нигде, поэтому
текст ниже ссылается на имена, а не на значения, и не может разойтись с кодом.*

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

REPO = Path.cwd().parent
DERIVED = REPO / "data" / "derived"
RAW_VR = REPO / "data" / "raw" / "vr"

SR = 90          # Hz after resampling
T_TRIAL = 210    # s, full recording

MIN_DURATION = 209.0    # s, a shorter recording is dropped
MAX_ZERO_FRAC = 0.5     # marker channel flat at zero for more than half the samples
MAX_CALIB_DRIFT = 0.10  # m, difference in mean HMD height between conditions

CONDITIONS = []
for c in ["HB", "LB"]:
    for t in [1, 2, 3, 4]:
        CONDITIONS.append(f"{c}_t{t}")

### Participant list

`TrialOrder_Part2.xlsx` is the roster: one row per participant with body
measures and the file index used for each condition. Taking the IDs from there
rather than hard-coding them keeps the notebook honest if the roster changes.

*`TrialOrder_Part2.xlsx` — список участников с антропометрией и индексами файлов
по условиям. Берём ID оттуда, а не хардкодим, чтобы блокнот не разошёлся с
реальностью при изменении списка.*

In [2]:
order = pd.read_excel(DERIVED / "TrialOrder_Part2.xlsx")
PARTICIPANTS = order["Subject ID"].dropna().tolist()

print(f"{len(PARTICIPANTS)} participants x {len(CONDITIONS)} conditions "
      f"= {len(PARTICIPANTS) * len(CONDITIONS)} cells")

14 participants x 8 conditions = 112 cells


### Coverage of the analysis table

Input is the wide table, one row per participant and six sway parameters per
condition. Counting how many participant × condition cells carry a value with
`.notna()` over the eight `PeriodicPower` columns, and listing the ones that do not.

*На входе широкая таблица: строка на участника, шесть параметров на условие.
Считаем через `.notna()` по восьми колонкам `PeriodicPower`, в скольких ячейках
есть значение, и перечисляем пустые.*

In [3]:
balance = pd.read_csv(DERIVED / "balance_data_2026.csv")

coverage = balance.set_index("ID")[[f"PeriodicPower {c}" for c in CONDITIONS]].notna()
coverage.columns = CONDITIONS

empty = []
for pid in coverage.index:
    for c in CONDITIONS:
        if not coverage.loc[pid, c]:
            empty.append((pid, c))

print(f"cells: {coverage.size} | filled: {coverage.size - len(empty)} | empty: {len(empty)}")
for pid, c in empty:
    print(f"  empty: {pid} {c}")

cells: 112 | filled: 107 | empty: 5
  empty: DO11UB30 HB_t1
  empty: DO11UB30 HB_t2
  empty: DO11UB30 HB_t3
  empty: DO11UB30 HB_t4
  empty: DO11UB30 LB_t4


### Is `rms_total` an independent measure?

`run_VRApp_analysis_2026.m` line 75 builds the resultant sway as
`sqrt(cdiff(com_ap).^2 + cdiff(com_ap).^2)` — the AP component twice, where AP
and ML were intended. If that is what ran, every `rms_total` is exactly
`sqrt(2) * rms_ap`. Testing it by dividing the two columns and comparing with
`np.allclose` rather than assuming.

*В строке 75 MATLAB-скрипта результирующее колебание собрано из `com_ap` дважды
вместо AP и ML. Если так, то `rms_total` тождественно равен `sqrt(2)·rms_ap` —
проверяем делением колонок и `np.allclose`, а не на слово.*

In [4]:
ratio = []
for c in CONDITIONS:
    ratio.append(balance[f"rms_total {c}"] / balance[f"rms_ap {c}"])
ratio = pd.concat(ratio).dropna()

print(f"rms_total / rms_ap over {len(ratio)} cells: "
      f"min {ratio.min():.6f}, max {ratio.max():.6f}, sqrt(2) = {np.sqrt(2):.6f}")
print("identical to sqrt(2) everywhere:", bool(np.allclose(ratio, np.sqrt(2))))

rms_total / rms_ap over 107 cells: min 1.414214, max 1.414214, sqrt(2) = 1.414214
identical to sqrt(2) everywhere: True


### Do repeated pipeline runs agree?

`VR-App_output.csv` is appended to on every run, so a trial processed more than
once appears more than once. Grouping by `fname` and counting distinct values per
column with `.nunique()` separates the two directly measured quantities from the
parameters that come out of the model fit.

*`VR-App_output.csv` дописывается при каждом прогоне, поэтому трайл может
встречаться несколько раз. Группируем по `fname` и считаем `.nunique()` по
колонкам, чтобы отделить измеренные величины от параметров подгонки модели.*

In [5]:
vr_log = pd.read_csv(DERIVED / "VR-App_output.csv")

measured = ["response sway power", "random sway power"]
fitted = ["visual Weight - W", "time delay - dt", "Loop Gain - Kp", "Kd",
          "Torque FB gain - Glp", "b", "sim Err"]

repeatability = []
for c in measured + fitted:
    g = vr_log.groupby("fname")[c]
    spread = (g.max() - g.min()) / g.mean().abs()
    repeatability.append({"parameter": c,
                          "trials with >1 value": int((g.nunique() > 1).sum()),
                          "max spread": spread.max()})
repeatability = pd.DataFrame(repeatability).set_index("parameter")

counts = sorted(vr_log["fname"].value_counts().unique())
print(f"{len(vr_log)} rows, {vr_log['fname'].nunique()} unique trials, repeat counts {counts}\n")
print(repeatability.to_string(float_format="{:.1%}".format))

387 rows, 107 unique trials, repeat counts [np.int64(2), np.int64(3), np.int64(4), np.int64(6)]

                      trials with >1 value  max spread
parameter                                             
response sway power                      0        0.0%
random sway power                        0        0.0%
visual Weight - W                       79       16.0%
time delay - dt                         79        3.0%
Loop Gain - Kp                          79        1.3%
Kd                                      79        3.8%
Torque FB gain - Glp                    79      300.0%
b                                       79       12.6%
sim Err                                 78        0.1%


### Collapsing the repeats

`pcl_ICfit_ml.m` optimises with `GlobalSearch`, which is stochastic, so repeated
rows cannot simply be assumed identical. Collapsing them with
`.groupby("fname").agg(median)`: a no-op for anything the pipeline measured, and a
stable choice for anything it fitted.

*`pcl_ICfit_ml.m` оптимизирует стохастическим `GlobalSearch`, поэтому считать
повторы одинаковыми нельзя. Схлопываем через `.groupby("fname").agg(median)` —
для измеренных величин это ничего не меняет, для подогнанных даёт устойчивый выбор.*

In [6]:
numeric = vr_log.select_dtypes("number").columns.tolist()

trial_log = vr_log.groupby("fname", as_index=False)[numeric].median()
trial_log["ID"] = vr_log.groupby("fname")["ID"].first().values

print(f"{len(vr_log)} rows -> {len(trial_log)} trials after collapsing repeats by median")

387 rows -> 107 trials after collapsing repeats by median


### Raw trial inventory

One row per raw recording of a main-sample participant. Duration comes from the
last line of the file via `seek()` to the tail, marker health from the first 30 s
via `read_csv(nrows=...)` — neither needs a full pass over 0.9 GB, and neither
writes to `data/raw/`.

*Строка на каждую сырую запись участников основной выборки. Длительность берём из
последней строки файла через `seek()` в хвост, состояние маркеров — из первых
30 с через `read_csv(nrows=...)`. Полный проход по 0,9 ГБ не нужен, в `data/raw/`
ничего не пишется.*

In [7]:
FOLDER = re.compile(r"^s(?P<pid>[A-Z0-9ÄÖÜ]+)_(?P<session>[AB])_(?P<cond>H1?B|LB1?)_*$")

records = []
for folder in tqdm(sorted(RAW_VR.iterdir())):
    match = FOLDER.match(folder.name) if folder.is_dir() else None
    if not match or match["pid"] not in PARTICIPANTS:
        continue

    for path in sorted(folder.glob("*.csv")):
        head = pd.read_csv(path, nrows=30 * SR,
                           usecols=["time", "ypos", "shld_ypos", "hip_ypos"])

        # duration = time column of the last row, read from the tail of the file
        with path.open("rb") as fh:
            fh.seek(max(0, path.stat().st_size - 4096))
            last = fh.read().splitlines()[-1].decode("utf-8", "replace")

        records.append({
            "ID": match["pid"],
            "session": match["session"],
            "condition": "HB" if match["cond"].startswith("H") else "LB",
            "file": path.name,
            "duration_s": float(last.split(",")[0]),
            "hmd_height_m": head["ypos"].mean(),
            "shoulder_height_m": head["shld_ypos"].mean(),
            "hip_height_m": head["hip_ypos"].mean(),
            "shoulder_zero_frac": (head["shld_ypos"] == 0).mean(),
            "hip_zero_frac": (head["hip_ypos"] == 0).mean(),
        })

inventory = pd.DataFrame(records).sort_values(["ID", "condition", "file"]).reset_index(drop=True)
inventory.to_csv(DERIVED / "qc_trial_inventory.csv", index=False)
print(f"{len(inventory)} recordings from {inventory['ID'].nunique()} participants")

  0%|          | 0/33 [00:00<?, ?it/s]

  9%|▉         | 3/33 [00:00<00:02, 11.24it/s]

 15%|█▌        | 5/33 [00:00<00:04,  5.80it/s]

 18%|█▊        | 6/33 [00:01<00:05,  4.93it/s]

 21%|██        | 7/33 [00:01<00:05,  4.64it/s]

...

 85%|████████▍ | 28/33 [00:08<00:02,  1.84it/s]

 91%|█████████ | 30/33 [00:11<00:03,  1.15s/it]

 94%|█████████▍| 31/33 [00:15<00:03,  1.74s/it]

100%|██████████| 33/33 [00:15<00:00,  2.16it/s]

117 recordings from 14 participants


### Two failure modes in the inventory

Filtering the inventory with boolean masks on `duration_s` and on the two
zero-fraction columns: a recording that stops early, and a marker channel that
was never written. The second one matters because `getCOM` needs both the hip and
the shoulder marker to reconstruct the centre of mass.

*Фильтруем инвентарь булевыми масками по `duration_s` и по долям нулей: запись,
оборвавшаяся раньше срока, и канал маркера, который вообще не писался. Второе
важно, потому что `getCOM` восстанавливает центр масс по маркерам таза и плеч.*

In [8]:
short = inventory[inventory["duration_s"] < MIN_DURATION]
print(f"shorter than {MIN_DURATION} s:")
print(short[["ID", "condition", "file", "duration_s"]].to_string(index=False))

inventory["flat"] = ((inventory["shoulder_zero_frac"] > MAX_ZERO_FRAC)
                     | (inventory["hip_zero_frac"] > MAX_ZERO_FRAC))

print("\nbody markers flat at zero:")
print(inventory[inventory["flat"]]
      .groupby(["ID", "condition"])[["shoulder_zero_frac", "hip_zero_frac"]]
      .agg(["size", "mean"]).to_string())

shorter than 209.0 s:
      ID condition                                                       file  duration_s
BI20OE24        HB Screen_Balance and VR_1_sBI20OE24_B_H1B__t1_INCOMPLETE.csv  181.424744
BI20OE24        LB   Screen_Balance and VR_1_sBI20OE24_A_LB_t1_INCOMPLETE.csv  161.123047
CA04TU11        HB   Screen_Balance and VR_1_sCA04TU11_B_HB_t1_INCOMPLETE.csv   75.347960
PE16IN18        LB   Screen_Balance and VR_1_sPE16IN18_A_LB_t1_INCOMPLETE.csv  203.635376

body markers flat at zero:
                   shoulder_zero_frac      hip_zero_frac     
                                 size mean          size mean
ID       condition                                           
AN06AN18 HB                         5  0.0             5  1.0
         LB                         4  0.0             4  1.0
DO11UB30 HB                         4  1.0             4  1.0


### Calibration drift between the two sessions

The centre of mass is scaled by the measured marker heights, so a participant
whose VR floor calibration differs between sessions carries an offset in exactly
the contrast of interest. Comparing mean HMD height per condition with
`.groupby().unstack()` and taking the absolute difference. Drift above
`MAX_CALIB_DRIFT` does not remove anyone: it is recorded as a flag and reported
as a limitation.

*Центр масс масштабируется измеренными высотами маркеров, поэтому расхождение
калибровки между сессиями даёт смещение ровно в интересующем контрасте.
Сравниваем среднюю высоту шлема по условиям через `.groupby().unstack()` и берём
модуль разности. Превышение `MAX_CALIB_DRIFT` никого не удаляет — оно
записывается как пометка и попадает в ограничения работы.*

In [9]:
calibration = inventory.groupby(["ID", "condition"])["hmd_height_m"].mean().unstack()
calibration["drift_m"] = (calibration["HB"] - calibration["LB"]).abs()
calibration = calibration.sort_values("drift_m", ascending=False)

print(calibration.round(3).to_string())

condition     HB     LB  drift_m
ID                              
EL30AD28   1.321  1.647    0.325
BI20OE24   1.484  1.493    0.009
CH10AL22   1.826  1.818    0.008
AN07IE11   1.667  1.661    0.006
AN23BE15   1.701  1.708    0.006
CH11RE22   1.567  1.561    0.006
CH08TU30   1.727  1.732    0.005
DO11UB30   1.749  1.753    0.004
RE13ON18   1.688  1.685    0.003
DI16UB31   1.517  1.521    0.003
AN06AN18   0.781  0.783    0.003
KA14RE15   1.608  1.607    0.001
PE16IN18   1.568  1.568    0.001
CA04TU11   1.681  1.682    0.000


### Assembling the QC table

Every rule is applied here and stored with the reason it fired, so the table in
the paper is generated rather than typed. The `action` column separates the two
kinds of finding: `exclude` drops the row from the analysis set, `keep-and-report`
leaves it in and sends it to the limitations. A trial-level rule only fires for
files the pipeline actually used — a recording that `TrialOrder_Part2.xlsx`
already routes around is not an exclusion.

The flat-marker rule is evaluated per condition first, with `.groupby().all()`, and
only promoted to the participant level when it fires in both. A marker channel that
failed for one podcast and not the other costs that participant the within-subject
contrast without making the other condition unusable, and the table has to say so
in the row rather than leave the gap unexplained.

*Каждое правило применяется здесь и сохраняется с причиной срабатывания, чтобы
таблица в статье генерировалась, а не набиралась руками. Колонка `action`
разделяет два типа находок: `exclude` убирает из выборки, `keep-and-report`
оставляет и отправляет в ограничения. Правило уровня трайла срабатывает только на
файлы, которые пайплайн реально брал.*

*Правило «плоского маркера» сначала считается по условиям через `.groupby().all()`
и повышается до уровня участника, только если сработало в обоих. Канал маркера,
отказавший на одном подкасте и не отказавший на другом, лишает участника
внутрисубъектного сравнения, но не делает второе условие непригодным, — и таблица
должна сказать это строкой, а не оставить пропуск без объяснения.*

In [10]:
qc = []

# Flat marker channel: first per condition, then promoted to the participant level
# only if every condition of that participant failed.
flat_cond = inventory.groupby(["ID", "condition"])["flat"].all()
flat_cond = flat_cond[flat_cond].reset_index()
n_cond = inventory.groupby("ID")["condition"].nunique()

for pid in flat_cond["ID"].unique():
    conds = flat_cond[flat_cond["ID"] == pid]["condition"].tolist()
    if len(conds) == n_cond[pid]:
        qc.append({"level": "participant", "ID": pid, "condition": "", "block": "", "file": "",
                   "action": "exclude",
                   "reason": "body markers flat at zero in every recording; "
                             "centre of mass not computable"})
    else:
        for cond in conds:
            qc.append({"level": "condition", "ID": pid, "condition": cond, "block": "", "file": "",
                       "action": "exclude",
                       "reason": f"body markers flat at zero in every {cond} recording; "
                                 f"centre of mass not computable in this condition"})

drift = calibration[calibration["drift_m"] > MAX_CALIB_DRIFT]
for pid in drift.index:
    qc.append({"level": "participant", "ID": pid, "condition": "", "block": "", "file": "",
               "action": "keep-and-report",
               "reason": f"VR calibration differs by {drift.loc[pid, 'drift_m']:.2f} m between "
                         f"conditions; absolute sway scaling not comparable"})

used = set(trial_log["fname"])
for _, r in short.iterrows():
    if r["file"] not in used:
        continue
    qc.append({"level": "trial", "ID": r["ID"], "condition": r["condition"],
               "block": int(re.search(r"_t(\d)", r["file"]).group(1)),
               "file": r["file"], "action": "exclude",
               "reason": f"recording ends at {r['duration_s']:.1f} s of {T_TRIAL} s; "
                         f"resampling would extrapolate"})

# Which participant x condition x block cells the rules above already account for.
covered = set()
for r in qc:
    if r["action"] != "exclude":
        continue
    if r["level"] == "participant":
        for c in CONDITIONS:
            covered.add((r["ID"], c))
    elif r["level"] == "condition":
        for c in CONDITIONS:
            if c.startswith(r["condition"]):
                covered.add((r["ID"], c))
    else:
        covered.add((r["ID"], f"{r['condition']}_t{r['block']}"))

# Anything still empty had no recording at all, which no rule above can see.
for pid, cell in empty:
    if (pid, cell) in covered:
        continue
    qc.append({"level": "trial", "ID": pid, "condition": cell.split("_t")[0],
               "block": int(cell.split("_t")[1]), "file": "", "action": "exclude",
               "reason": "no recording for this block in the raw data"})

qc = pd.DataFrame(qc)[["level", "ID", "condition", "block", "file", "action", "reason"]]
qc.to_csv(DERIVED / "qc_exclusions.csv", index=False)
print(qc.drop(columns="file").to_string(index=False))

      level       ID condition block          action                                                                                           reason
participant AN06AN18                         exclude                      body markers flat at zero in every recording; centre of mass not computable
  condition DO11UB30        HB               exclude body markers flat at zero in every HB recording; centre of mass not computable in this condition
participant EL30AD28                 keep-and-report        VR calibration differs by 0.33 m between conditions; absolute sway scaling not comparable
      trial BI20OE24        HB     1         exclude                                 recording ends at 181.4 s of 210 s; resampling would extrapolate
      trial BI20OE24        LB     1         exclude                                 recording ends at 161.1 s of 210 s; resampling would extrapolate
      trial PE16IN18        LB     1         exclude                                 recording ends 

### Resulting analysis set

Subtracting only the participants marked `exclude` from the roster, then counting
how many of the survivors have data in both conditions — that subset is what a
within-subject contrast can use. Flagged participants stay in.

*Вычитаем из списка только тех, кто помечен `exclude`, и считаем, у скольких из
оставшихся есть данные в обоих условиях: этот набор доступен внутрисубъектному
сравнению. Помеченные участники остаются в выборке.*

In [11]:
dropped = qc[(qc["level"] == "participant") & (qc["action"] == "exclude")]["ID"].tolist()
flagged = qc[qc["action"] == "keep-and-report"]["ID"].tolist()
half = [p for p in qc[qc["level"] == "condition"]["ID"].unique() if p not in dropped]

analysis_set = []
paired = []
for p in PARTICIPANTS:
    if p in dropped:
        continue
    analysis_set.append(p)
    has_hb = coverage.loc[p, ["HB_t1", "HB_t2", "HB_t3", "HB_t4"]].any()
    has_lb = coverage.loc[p, ["LB_t1", "LB_t2", "LB_t3", "LB_t4"]].any()
    if has_hb and has_lb and p not in half:
        paired.append(p)

print(f"recruited                     {len(PARTICIPANTS)}")
print(f"excluded                      {len(dropped)}  ({', '.join(sorted(dropped))})")
print(f"analysis set                  {len(analysis_set)}")
print(f"  of which flagged            {len(flagged)}  ({', '.join(sorted(flagged))})")
print(f"  losing one whole condition  {len(half)}  ({', '.join(sorted(half))})")
print(f"with data in both conditions  {len(paired)}")
print(f"trial-level exclusions        {int((qc['level'] == 'trial').sum())}")

recruited                     14
excluded                      1  (AN06AN18)
analysis set                  13
  of which flagged            1  (EL30AD28)
  losing one whole condition  1  (DO11UB30)
with data in both conditions  12
trial-level exclusions        4
